In [ ]:
# =============================================================
#  Importación de librerías y carga de datasets
# =============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar los datasets
original = pd.read_csv("../datasets/input/turkish_music_emotion_original.csv")
modified = pd.read_csv("../datasets/input/turkish_music_emotion_modified.csv")

print("Shape original:", original.shape)
print("Shape modified:", modified.shape)

In [ ]:
# =============================================================
# Exploración inicial (EDA básico)
# =============================================================
print("\nInformación del dataset original:")
print(original.info())

print("\nInformación del dataset modificado:")
print(modified.info())


# Conteo de valores nulos
def resumen_nulos(df):
    nulos = df.isnull().sum()
    return nulos[nulos > 0].sort_values(ascending=False)


print("\nValores nulos en el dataset modificado:")
print(resumen_nulos(modified))

In [ ]:
# =============================================================
# Análisis de inconsistencias de tipo
# =============================================================

# Detección de columnas tipo 'object' que deberían ser numéricas
object_cols = [col for col in modified.columns if modified[col].dtype == "object" and col != "Class"]
print(f"\nColumnas con tipos 'object' sospechosos: {len(object_cols)}")
print(object_cols[:10])


# Ejemplo de valores únicos para identificar errores
def valores_unicos(df, cols, limit=5):
    for c in cols[:limit]:
        print(f"\n>>> Columna: {c}")
        print(df[c].unique()[:10])


valores_unicos(modified, object_cols)

In [ ]:
# =============================================================
# Conversión de columnas numéricas y manejo de errores
# =============================================================

# Convertir todas las columnas excepto 'Class'
columnas_numericas = modified.columns.difference(["Class", "mixed_type_col"])
modified[columnas_numericas] = modified[columnas_numericas].apply(pd.to_numeric, errors="coerce")

# Eliminar la columna
modified = modified.drop("mixed_type_col", axis=1, errors="ignore")

# Revisión de nulos post conversión
print("\nValores nulos después de conversión a numéricos:")
print(resumen_nulos(modified))

In [ ]:
# =============================================================
# Corrección de valores mal formateados en columnas numéricas
# =============================================================

for col in modified.columns:
    if col != "Class":
        # Convertir a string temporalmente
        modified[col] = modified[col].astype(str)
        # Quitar comas al final o caracteres extraños
        modified[col] = modified[col].str.replace(",", "", regex=False)
        # Intentar convertir nuevamente a numérico
        modified[col] = pd.to_numeric(modified[col], errors="coerce")

In [ ]:
# =============================================================
#  Limpieza de valores nulos
# =============================================================

# Estrategia: imputar valores numéricos con la mediana
for col in modified.columns:
    if modified[col].dtype in ["float64", "int64"]:
        modified[col].fillna(modified[col].median(), inplace=True)

# Eliminar filas sin clase
modified.dropna(subset=["Class"], inplace=True)

In [ ]:
# =============================================================
#  Detección y tratamiento de outliers
# =============================================================


# Método IQR (Interquartile Range)
def eliminar_outliers(df, columnas, k=3):
    for c in columnas:
        if df[c].dtype in ["float64", "int64"]:
            Q1 = df[c].quantile(0.25)
            Q3 = df[c].quantile(0.75)
            IQR = Q3 - Q1
            low = Q1 - k * IQR
            high = Q3 + k * IQR
            df = df[(df[c] >= low) & (df[c] <= high)]
    return df


modified_clean = eliminar_outliers(modified, modified.columns[1:])
print(f"\nDataset final sin outliers: {modified_clean.shape}")

In [ ]:
# =============================================================
# Normalización de la columna Class
# =============================================================

# Convertir todo a minúsculas
modified_clean["Class"] = modified_clean["Class"].str.strip().str.lower()

# Revisar distribución después de la corrección
print(modified_clean["Class"].value_counts())

In [ ]:
# =============================================================
#  Análisis exploratorio (visual)
# =============================================================

plt.figure(figsize=(10, 4))
sns.countplot(x="Class", data=modified_clean, palette="viridis")
plt.title("Distribución de Clases Emocionales")
plt.xticks(rotation=45)
plt.show()

# Correlación entre variables numéricas
corr = modified_clean.select_dtypes(include=["float64", "int64"]).corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Mapa de Correlación entre Variables Numéricas")
plt.show()

In [ ]:
# =============================================================
#  Exportación del dataset limpio
# =============================================================
modified_clean.to_csv("../datasets/output/EDA_cleaned.csv", index=False)
print("\n Dataset limpio exportado como 'EDA_cleaned.csv'")